In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta, time, date
import time 
from sqlalchemy import text
import oracledb
import sys

In [2]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [3]:
tabla='dat_defunciones_ggi'
inicioTiempo = time.time()
now_inicio = datetime.now()

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
		"host": hostname,
		"port": port,
		"service_name": service_name
	}
)

connection0 = engine0.connect()

query = f"""SELECT
x.COD_CENTRO, 
x.PERIODO, 
x.tipo_paciente, 
sum(x.CANTIDAD) AS CANTIDAD from (
	SELECT 
    t.DEFCENASICOD as COD_CENTRO, 
    TO_CHAR(t.DEFHORFEC,'yyyymm') AS PERIODO,
	(SELECT 
    CASE WHEN e1.TIPOPACICOD = '4' THEN '0' ELSE '1' END  FROM SGSS.CBTPC10 e1 
    WHERE am.actmedtipopacicod = e1.tipopacicod) AS TIPO_PACIENTE, 1 AS CANTIDAD
	FROM SGSS.CTDEF10 t
	LEFT OUTER JOIN CMAME10 am ON t.defpacsecnum = am.actmedpacsecnum
		AND t.DEFORICENASICOD = am.oricenasicod
		AND t.defcenasicod    = am.cenasicod
		AND t.defactmednum    = am.actmednum
	WHERE
	TO_CHAR(t.DEFHORFEC,'yyyy') = '2024' and
	t.DEFAREHOSCOD = '02' AND
	t.DEFESTREGCOD = '1') x
	group by x.COD_CENTRO , x.PERIODO, x.tipo_paciente
"""

base1 = pd.read_sql_query(query, con=connection0)

connection0.close()

base1.to_sql(name=f'{tabla}', con=engine2, if_exists='replace', index=False)

finproceso=time.time()
tiempoproceso=finproceso - inicioTiempo
tiempoproceso=round(tiempoproceso,3)
print("Proceso completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso completado satisfactoriamente en 7.12 segundos
